# Imports

In [1]:
import os
import sys

project_root = "Incremental-Learning-MycroftMinds"
cwd = os.path.abspath(os.getcwd())
while True:
    if os.path.basename(cwd) == project_root:
        if cwd not in sys.path:
            sys.path.insert(0, cwd)
        break
    parent = os.path.dirname(cwd)
    if parent == cwd:
        raise FileNotFoundError(f"Project root '{project_root}' not found in any parent directory.")
    cwd = parent

In [2]:
import time

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from river.forest import ARFRegressor
from river.tree import HoeffdingTreeRegressor
from src.bagging import ARFRegressorFODT
from src.trees import fastodt
from src.boosting import IncubationBoost
from src.ensemble_methods import elasticGBDT
from src.evaluation.baseline import PreviousValueRegressor

from src.data.io import load_dataset
from src.evaluation.tuning import hyperparameter_search_tuning
from src.data import dataloader
from src.utils.logging import load_checkpoint_results, list_log_files, view_log_tail

# Load the data

In [3]:
# check you data dir and file prefixes
data_dir = '../../data/second_batch/commercial'

AGGREGATION = 'hourly'  # only affects the name of the aggregation, can be human-readable
DATA_FREQUENCY = 'H'  # 1-Hour frequency
FORECAST_HORIZON = '1h'  # 1-Hour forecast horizon
SPIKE_PROMINENCE = 10  # Prominence for spike detection, adjusted after EDA
USE_RESIDUAL = True  # If True, model the residuals (1-step differencing) instead of the original target.

# Measure loading time
start_time = time.time()

# Load data
train, valid, test = load_dataset(data_dir=data_dir, prefix='sb_com_hour_wfv', use_numpy=True)
FEATURES = ['time', 'device_id', 'hour', 'weekday', 'month',
           'consumption_lag_1h', 'consumption_lag_2h', 'consumption_lag_3h', 'consumption_lag_24h']

loading_time = time.time() - start_time
print(f"Data loaded in {loading_time/60:.2f} minutes")

[INFO] Loading train data...
[INFO] Train data loaded: 2 samples
[INFO] Loading valid data...
[INFO] Valid data loaded: 2 samples
[INFO] Loading test data...
[INFO] Test data loaded: 2 samples
Data loaded in 0.00 minutes


# Hyperparameter tuning

The enhanced grid search will automatically handle logging and checkpoint creation with custom file prefixes:
- Creates log directories automatically with sensible defaults
- Uses custom file prefixes (e.g., `sb_com_1h_wfv`) for organized file naming
- Logs all parameter combinations and results to timestamped files
- Saves intermediate results to prevent data loss
- Provides detailed progress tracking

File naming pattern: `timestamp_prefix_modelname_tuning.txt` and `prefix_modelname_results.pkl`

## Configuration

Set up experiment configuration and file naming.

In [5]:
# Experiment configuration
EXPERIMENT_PREFIX = "1h_wfv"  # This will be used in all result filenames
RESIDUAL_PREFIX = ("resid_" if USE_RESIDUAL else "")  # if we use residuals
BATCH_MODELS_LOG_PREFIX = RESIDUAL_PREFIX + "sb_com_batch_models"
INCREMENTAL_MODELS_LOG_PREFIX = RESIDUAL_PREFIX + "sb_com_incremental_models"

print(f"Experiment prefix: {EXPERIMENT_PREFIX}")
print(f"Batch models will be saved to: .../results/{BATCH_MODELS_LOG_PREFIX}/")
print(f"Incremental models will be saved to: .../results/{INCREMENTAL_MODELS_LOG_PREFIX}/")
print(f"Files will be named with pattern: {EXPERIMENT_PREFIX}_ModelName_results.pkl")

Experiment prefix: 1h_wfv
Batch models will be saved to: .../results/resid_sb_com_batch_models/
Incremental models will be saved to: .../results/resid_sb_com_incremental_models/
Files will be named with pattern: 1h_wfv_ModelName_results.pkl


## Batch-Learning Models

### GBDT

In [ ]:
update_freq = '14D'  # Batch learning update frequency

# GBDT hyperparameter tuning
gbdt = GradientBoostingRegressor

param_grid_gbdt = {
    'n_estimators': [25, 50, 100, 150, 200],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'max_depth': [3, 7, 12, 15],
    'subsample': [0.6, 0.8, 1.0],
    'random_state': [42]
}

print("Starting GBDT hyperparameter tuning...")

best_params_gbdt = hyperparameter_search_tuning(
    param_grid_gbdt,
    gbdt,
    train,
    valid,
    use_residuals=USE_RESIDUAL,
    spike_prominence=SPIKE_PROMINENCE,
    aggregation=AGGREGATION,
    data_freq=DATA_FREQUENCY,
    forecast_horizon=FORECAST_HORIZON,
    features=FEATURES,
    update_freq=update_freq,  # Batch learning update frequency
    search_type='random',
    n_iter=100,
    log_prefix=BATCH_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX,
    save_intermediate=True
)

In [24]:
print("GBDT Best Parameters:")
print(best_params_gbdt)

GBDT Best Parameters:
{'overall': {'best_params': {'rmse': {'n_estimators': 50, 'learning_rate': 0.1, 'max_depth': 3, 'subsample': 1.0, 'random_state': 42}, 'mae': {'n_estimators': 200, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 0.6, 'random_state': 42}, 'mape': {'n_estimators': 200, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 0.6, 'random_state': 42}, 'dir_accuracy': {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 3, 'subsample': 0.6, 'random_state': 42}, 'first_diff_corr': {'n_estimators': 50, 'learning_rate': 0.1, 'max_depth': 3, 'subsample': 1.0, 'random_state': 42}, 'spikes_immediate.f1': {'n_estimators': 50, 'learning_rate': 0.01, 'max_depth': 15, 'subsample': 1.0, 'random_state': 42}, 'drops_immediate.f1': {'n_estimators': 50, 'learning_rate': 0.05, 'max_depth': 15, 'subsample': 1.0, 'random_state': 42}}, 'best_scores': {'rmse': 0.020841983678335493, 'mae': 0.014608867867345467, 'mape': 0.03291787231790834, 'dir_accuracy': 0.5733304387622188, 'first_di

### Random Forest

In [ ]:
update_freq = '14D'  # Batch learning update frequency

# Random Forest hyperparameter tuning
rf = RandomForestRegressor

param_grid_rf = {
    'n_estimators': [50, 100, 150, 200, 300],
    'max_depth': [3, 5, 7, 10, 15, 20],
    'max_features': [0.3, 0.5, 0.7, 1.0],
    'random_state': [42],
    'n_jobs': [-1]
}

print("Starting Random Forest hyperparameter tuning...")
best_params_rf = hyperparameter_search_tuning(
    param_grid_rf,
    rf,
    train,
    valid,
    use_residuals=USE_RESIDUAL,
    spike_prominence=SPIKE_PROMINENCE,
    aggregation=AGGREGATION,
    data_freq=DATA_FREQUENCY,
    forecast_horizon=FORECAST_HORIZON,
    features=FEATURES,
    update_freq=update_freq,  # Batch learning update frequency
    search_type='random',
    n_iter=100,
    log_prefix=BATCH_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX
)

Starting Random Forest hyperparameter tuning...


Grid Search Progress:   0%|          | 0/100 [00:00<?, ?it/s]

Results saved to ../results/batch_models_results/sb_com_1h_wfv_RandomForestRegressor_intermediate_results.pkl
Results saved to ../results/batch_models_results/sb_com_1h_wfv_RandomForestRegressor_intermediate_per_device_results.pkl
Results saved to ../results/batch_models_results/sb_com_1h_wfv_RandomForestRegressor_intermediate_results.pkl
Results saved to ../results/batch_models_results/sb_com_1h_wfv_RandomForestRegressor_intermediate_per_device_results.pkl
Results saved to ../results/batch_models_results/sb_com_1h_wfv_RandomForestRegressor_intermediate_results.pkl
Results saved to ../results/batch_models_results/sb_com_1h_wfv_RandomForestRegressor_intermediate_per_device_results.pkl
Results saved to ../results/batch_models_results/sb_com_1h_wfv_RandomForestRegressor_intermediate_results.pkl
Results saved to ../results/batch_models_results/sb_com_1h_wfv_RandomForestRegressor_intermediate_per_device_results.pkl
Results saved to ../results/batch_models_results/sb_com_1h_wfv_RandomForestR

In [33]:
print("Random Forest Best Parameters:")
print(best_params_rf)

Random Forest Best Parameters:
{'overall': {'best_params': {'rmse': {'n_estimators': 150, 'max_depth': 5, 'max_features': 0.5, 'random_state': 42, 'n_jobs': -1}, 'mae': {'n_estimators': 200, 'max_depth': 3, 'max_features': 1.0, 'random_state': 42, 'n_jobs': -1}, 'mape': {'n_estimators': 150, 'max_depth': 3, 'max_features': 0.7, 'random_state': 42, 'n_jobs': -1}, 'dir_accuracy': {'n_estimators': 300, 'max_depth': 15, 'max_features': 0.3, 'random_state': 42, 'n_jobs': -1}, 'first_diff_corr': {'n_estimators': 100, 'max_depth': 3, 'max_features': 0.7, 'random_state': 42, 'n_jobs': -1}, 'spikes_immediate.f1': {'n_estimators': 100, 'max_depth': 3, 'max_features': 0.7, 'random_state': 42, 'n_jobs': -1}, 'drops_immediate.f1': {'n_estimators': 150, 'max_depth': 5, 'max_features': 1.0, 'random_state': 42, 'n_jobs': -1}}, 'best_scores': {'rmse': 0.02033992611444141, 'mae': 0.014158158663364503, 'mape': 0.03189849220594684, 'dir_accuracy': 0.5757771336705118, 'first_diff_corr': 0.2632978584881213,

## Incremental Learning Models

### HoeffdingTree

In [ ]:
update_freq = "1h"

hoef = HoeffdingTreeRegressor

param_grid = {
    'max_depth': [3, 5, 10, 15, 20],
    'grace_period': [0, 10, 24, 48, 69, 96],
    'tau': [0.001, 0.005, 0.01, 0.03, 0.05, 0.1],
    'delta': [0.001, 0.005, 0.01, 0.05, 0.1],
    #'merit_preprune': [True, False]
}

print("Starting HoeffdingTree hyperparameter tuning...")
best_params_hf = hyperparameter_search_tuning(
    param_grid,
    hoef,
    train,
    valid,
    use_residuals=USE_RESIDUAL,
    spike_prominence=SPIKE_PROMINENCE,
    aggregation=AGGREGATION,
    data_freq=DATA_FREQUENCY,
    forecast_horizon=FORECAST_HORIZON,
    features=FEATURES,
    update_freq=update_freq,
    search_type='random',
    n_iter=100,
    log_prefix=INCREMENTAL_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX
)

Starting HoeffdingTree hyperparameter tuning...


Grid Search Progress:   0%|          | 0/100 [00:00<?, ?it/s]

Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_HoeffdingTreeRegressor_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_HoeffdingTreeRegressor_intermediate_per_device_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_HoeffdingTreeRegressor_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_HoeffdingTreeRegressor_intermediate_per_device_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_HoeffdingTreeRegressor_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_HoeffdingTreeRegressor_intermediate_per_device_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_HoeffdingTreeRegressor_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_HoeffdingTreeRegressor_intermediate_per_device_results.

In [28]:
print("HoeffdingTree Best Parameters:")
print(best_params_hf)

HoeffdingTree Best Parameters:
{'overall': {'best_params': {'rmse': {'max_depth': 10, 'grace_period': 48, 'tau': 0.1, 'delta': 0.005}, 'mae': {'max_depth': 10, 'grace_period': 48, 'tau': 0.1, 'delta': 0.005}, 'mape': {'max_depth': 10, 'grace_period': 48, 'tau': 0.1, 'delta': 0.005}, 'dir_accuracy': {'max_depth': 20, 'grace_period': 0, 'tau': 0.005, 'delta': 0.01}, 'first_diff_corr': {'max_depth': 10, 'grace_period': 24, 'tau': 0.001, 'delta': 0.1}, 'spikes_immediate.f1': {'max_depth': 3, 'grace_period': 10, 'tau': 0.005, 'delta': 0.001}, 'drops_immediate.f1': {'max_depth': 3, 'grace_period': 24, 'tau': 0.03, 'delta': 0.001}}, 'best_scores': {'rmse': 0.029402880407697354, 'mae': 0.022847965899188607, 'mape': 0.05246176170522797, 'dir_accuracy': 0.5332034687914383, 'first_diff_corr': 0.3598807211752353, 'spikes_immediate.f1': 0.10118861432593056, 'drops_immediate.f1': 0.12442141623488773}}, 'per_device': {'best_params_lower_better': {'sub2': {'rmse': {'max_depth': 3, 'grace_period': 69, 

### ARF-HoeffdingTree

In [ ]:
param_grid = {
    'n_models': [5, 10, 20, 40, 60, 80, 100],
    'max_features': [0.1, 0.3, 0.5, 0.7, 1.0],
    'max_depth': [5, 7, 10, 15, 25, 30, 50, 70, 100],
    'aggregation_method': ['mean', 'median'],
    'disable_weighted_vote': [True, False],
    'drift_detector': [None],
    'warning_detector': [None],
    'tau': [0.001, 0.005, 0.01, 0.03, 0.05, 0.1],
    'delta': [0.001, 0.005, 0.01, 0.05, 0.1],
    'grace_period': [0, 10, 24, 35, 48, 69, 96],
}

arf_hf = ARFRegressor

print("Starting ARF (Hoeffding Tree) hyperparameter tuning...")
best_params_arf_hf = hyperparameter_search_tuning(
    param_grid,
    arf_hf,
    train,
    valid,
    use_residuals=USE_RESIDUAL,
    spike_prominence=SPIKE_PROMINENCE,
    aggregation=AGGREGATION,
    data_freq=DATA_FREQUENCY,
    forecast_horizon=FORECAST_HORIZON,
    features=FEATURES,
    update_freq=update_freq,
    search_type='random',
    n_iter=100,
    log_prefix=INCREMENTAL_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX
)

Starting ARF (Hoeffding Tree) hyperparameter tuning...


Grid Search Progress:   0%|          | 0/100 [00:00<?, ?it/s]

Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_ARFRegressor_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_ARFRegressor_intermediate_per_device_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_ARFRegressor_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_ARFRegressor_intermediate_per_device_results.pkl


In [ ]:
print("ARF (Hoeffding Tree) Best Parameters:")
print(best_params_arf_hf)

### fastODT

In [ ]:
update_freq = "1h"  # for incremental models put the same as the data frequency to recreate real world scenario

fodt = fastodt.FastODT

param_grid = {
    'depth': [5, 10, 15, 30, 50, 100],
    'grace_period': [0, 10, 24, 48, 96, 168],
    'grace_period_depth': [-1, 0, 1],
    'split_confidence': [0.001, 0.005, 0.01, 0.05, 0.1, 0.5],
    'max_feature': [0.3, 0.5, 0.7, 1.0],
    'tie_threshold': [0.01, 0.05],
    'hist':[ 
        HistogramContainer(
            bin_type = get_histogram_type('adaptive'),
            max_bins = 8
        ),
        HistogramContainer(
            bin_type = get_histogram_type('adaptive'),
            max_bins = 16
        ),
        HistogramContainer(
            bin_type = get_histogram_type('adaptive'),
            max_bins = 32
        )
    ],
    'split_criterion': VarianceReductionRelative
}

print("Starting FastODT hyperparameter tuning...")
best_params_fastODT = hyperparameter_search_tuning(
    param_grid,
    fodt,
    train,
    valid,
    use_residuals=USE_RESIDUAL,
    spike_prominence=SPIKE_PROMINENCE,
    aggregation=AGGREGATION,
    data_freq=DATA_FREQUENCY,
    forecast_horizon=FORECAST_HORIZON,
    features=FEATURES,
    update_freq=update_freq,
    search_type='random',
    n_iter=100,
    log_prefix=INCREMENTAL_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX
)

Starting FastODT hyperparameter tuning...


Grid Search Progress:   0%|          | 0/100 [00:00<?, ?it/s]

Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_FastODT_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_FastODT_intermediate_per_device_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_FastODT_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_FastODT_intermediate_per_device_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_FastODT_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_FastODT_intermediate_per_device_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_FastODT_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_FastODT_intermediate_per_device_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_FastODT_intermediate_results.pkl
Results sav

In [18]:
print("FastODT Best Parameters:")
for key, value in best_params_fastODT.items():
    print(f"{key}: {value}")

FastODT Best Parameters:
overall: {'best_params': {'rmse': {'depth': 30, 'grace_period': 10, 'grace_period_depth': -1, 'split_confidence': 0.5, 'max_feature': 0.7, 'tie_threshold': 0.01, 'bin_type': 'adaptive', 'max_bins': 8, 'split_criterion': 'vr_rel'}, 'mae': {'depth': 100, 'grace_period': 24, 'grace_period_depth': 1, 'split_confidence': 0.5, 'max_feature': 0.3, 'tie_threshold': 0.01, 'bin_type': 'adaptive', 'max_bins': 32, 'split_criterion': 'vr_rel'}, 'mape': {'depth': 100, 'grace_period': 24, 'grace_period_depth': 1, 'split_confidence': 0.5, 'max_feature': 0.3, 'tie_threshold': 0.01, 'bin_type': 'adaptive', 'max_bins': 32, 'split_criterion': 'vr_rel'}, 'dir_accuracy': {'depth': 30, 'grace_period': 96, 'grace_period_depth': 0, 'split_confidence': 0.1, 'max_feature': 1.0, 'tie_threshold': 0.01, 'bin_type': 'adaptive', 'max_bins': 8, 'split_criterion': 'vr_rel'}, 'first_diff_corr': {'depth': 30, 'grace_period': 10, 'grace_period_depth': -1, 'split_confidence': 0.5, 'max_feature': 0.

### ARF-fastODT

In [ ]:
update_freq = "1h"

arf = ARFRegressorFODT

param_grid = {
    'n_models': [5, 60, 15, 30],
    'max_features': [0.7, 0.3, 0.5],
    'max_depth': [5, 30, 10],
    'aggregation_method': ['mean'],
    'disable_weighted_vote': [False],
    'drift_detector': [None],
    'warning_detector': [None],
    'eagerness': [0],
    'tau': [0.01, 0.05],
    'delta': [0.01, 0.5, 0.05, 0.1, 0.005],
    'grace_period': [0, 24, 48, 96],
    'grace_period_depth': [-1, 0, 1],
    'hist': 
        HistogramContainer(
            bin_type = get_histogram_type('mean')
        ),
    'split_criterion': VarianceReductionRelative
}

print("Starting ARF (FastODT) hyperparameter tuning...")
best_params_arf = hyperparameter_search_tuning(
    param_grid,
    arf,
    train,
    valid,
    use_residuals=USE_RESIDUAL,
    spike_prominence=SPIKE_PROMINENCE,
    aggregation=AGGREGATION,
    data_freq=DATA_FREQUENCY,
    forecast_horizon=FORECAST_HORIZON,
    features=FEATURES,
    update_freq=update_freq,
    search_type='random',
    n_iter=100,
    log_prefix=INCREMENTAL_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX
)

Starting ARF (FastODT) hyperparameter tuning...


Grid Search Progress:   0%|          | 0/100 [00:00<?, ?it/s]

Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_ARFRegressorFODT_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_ARFRegressorFODT_intermediate_per_device_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_ARFRegressorFODT_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_ARFRegressorFODT_intermediate_per_device_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_ARFRegressorFODT_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_ARFRegressorFODT_intermediate_per_device_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_ARFRegressorFODT_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_ARFRegressorFODT_intermediate_per_device_results.pkl
Results saved to ../results/incremental_mode

In [20]:
print("ARF (FastODT) Best Parameters:")
print(best_params_arf)

ARF (FastODT) Best Parameters:
{'overall': {'best_params': {'rmse': {'n_models': 5, 'max_features': 0.5, 'max_depth': 30, 'aggregation_method': 'mean', 'disable_weighted_vote': False, 'drift_detector': None, 'warning_detector': None, 'eagerness': 0, 'tau': 0.05, 'delta': 0.05, 'grace_period': 96, 'grace_period_depth': 0, 'bin_type': 'mean', 'split_criterion': 'vr_rel'}, 'mae': {'n_models': 5, 'max_features': 0.5, 'max_depth': 30, 'aggregation_method': 'mean', 'disable_weighted_vote': False, 'drift_detector': None, 'warning_detector': None, 'eagerness': 0, 'tau': 0.05, 'delta': 0.05, 'grace_period': 96, 'grace_period_depth': 0, 'bin_type': 'mean', 'split_criterion': 'vr_rel'}, 'mape': {'n_models': 15, 'max_features': 0.7, 'max_depth': 5, 'aggregation_method': 'mean', 'disable_weighted_vote': False, 'drift_detector': None, 'warning_detector': None, 'eagerness': 0, 'tau': 0.01, 'delta': 0.01, 'grace_period': 48, 'grace_period_depth': 0, 'bin_type': 'mean', 'split_criterion': 'vr_rel'}, 'd

### incubationBoost

In [ ]:
update_freq = "1h"  # for incremental models put the same as the data frequency to recreate real world scenario:

incb = IncubationBoost

param_grid = {
    
    'length': [5, 10, 20, 30, 40],
    'learning_rate': [0.001, 0.5, 0.01, 0.1],
    'depth': [5, 30, 10, 20],
    'grace_period': [0, 24, 48],
    'grace_period_depth': [-1, 1],
    'split_confidence': [0.01, 0.05, 0.1, 0.5],
    'max_feature': [0.3, 0.5, 0.7, 1.0],
    'tie_threshold': [0.01, 0.05],
    'hist':[ 
        HistogramContainer(
            bin_type = get_histogram_type('adaptive'),
            max_bins = 8
        ),
        HistogramContainer(
            bin_type = get_histogram_type('adaptive'),
            max_bins = 16
        ),
        HistogramContainer(
            bin_type = get_histogram_type('adaptive'),
            max_bins = 32
        )
    ],
    'split_criterion': VarianceReductionRelative,
    'verbose':[False]
}

print("Starting Incubation Boost hyperparameter tuning...")
best_params_inc = hyperparameter_search_tuning(
    param_grid,
    incb,
    train,
    valid,
    use_residuals=USE_RESIDUAL,
    spike_prominence=SPIKE_PROMINENCE,
    aggregation=AGGREGATION,
    data_freq=DATA_FREQUENCY,
    forecast_horizon=FORECAST_HORIZON,
    features=FEATURES,
    update_freq=update_freq,
    search_type='random',
    n_iter=100,
    log_prefix=INCREMENTAL_MODELS_LOG_PREFIX,
    file_prefix=EXPERIMENT_PREFIX
)

Starting Incubation Boost hyperparameter tuning...


Grid Search Progress:   0%|          | 0/100 [00:00<?, ?it/s]

Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_IncubationBoost_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_IncubationBoost_intermediate_per_device_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_IncubationBoost_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_IncubationBoost_intermediate_per_device_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_IncubationBoost_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_IncubationBoost_intermediate_per_device_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_IncubationBoost_intermediate_results.pkl
Results saved to ../results/incremental_models_v2_results/sb_com_1h_wfv_IncubationBoost_intermediate_per_device_results.pkl
Results saved to ../results/incremental_models_v2_re

In [22]:
print("Incubation Boost Best Parameters:")
for key, value in best_params_inc.items():
    print(f"{key}: {value}")

Incubation Boost Best Parameters:
overall: {'best_params': {'rmse': {'base_learner': <class 'src.trees.fastodt.FastODT'>, 'length': 40, 'learning_rate': 0.5, 'depth': 10, 'grace_period': 0, 'grace_period_depth': 1, 'split_confidence': 0.05, 'max_feature': 1.0, 'tie_threshold': 0.01, 'bin_type': 'adaptive', 'max_bins': 16, 'split_criterion': 'vr_rel', 'verbose': False}, 'mae': {'base_learner': <class 'src.trees.fastodt.FastODT'>, 'length': 40, 'learning_rate': 0.5, 'depth': 10, 'grace_period': 0, 'grace_period_depth': 1, 'split_confidence': 0.05, 'max_feature': 1.0, 'tie_threshold': 0.01, 'bin_type': 'adaptive', 'max_bins': 16, 'split_criterion': 'vr_rel', 'verbose': False}, 'mape': {'base_learner': <class 'src.trees.fastodt.FastODT'>, 'length': 40, 'learning_rate': 0.5, 'depth': 10, 'grace_period': 0, 'grace_period_depth': 1, 'split_confidence': 0.05, 'max_feature': 1.0, 'tie_threshold': 0.01, 'bin_type': 'adaptive', 'max_bins': 16, 'split_criterion': 'vr_rel', 'verbose': False}, 'dir_